# Adversarial Estimation of a Nonlinear Network Effort Game

Effort-game notebook following `docs/effort_game_estimation_design.md`.


## C1 - Imports, seeds, provenance capture


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import subprocess
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings(
    "ignore",
    message=r"`torch_geometric\.distributed` has been deprecated since 2\.7\.0.*",
    category=DeprecationWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"`torch\.jit\.script` is deprecated\..*",
    category=DeprecationWarning,
)

import torch_geometric
from torch_geometric.utils import degree, from_networkx, k_hop_subgraph, to_undirected

print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("networkx:", nx.__version__)
print("numpy:", np.__version__)

torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.float32)

DEVICE = torch.device("cpu")
print("device:", DEVICE)

# torch.use_deterministic_algorithms(True)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from constants import FIGURE_DPI
from plot_style import (
    add_reference_line,
    apply_paper_style,
    figure_size,
    line_style,
    scatter_style,
    series_style,
    style_axes,
    style_histogram_patches,
)

from effort_generator import EffortGameGenerator
from config import EffortExperimentConfig
from discriminator import RootedMPNNDiscriminator

from utils import (
    build_row_stochastic_W,
    compute_instance_noise_taus,
    extract_ego_batch,
)
from root_sampling import RootSampler, build_adjacency_from_edge_index, sample_roots_tensor

from visualization import (
    plot_ground_truth_graph_outcomes,
    plot_loss_convergence,
)

apply_paper_style()


torch: 2.10.0+cpu
torch_geometric: 2.7.0
networkx: 3.6.1
numpy: 1.26.4
device: cpu


## C2 - Configuration (single source of truth)


In [2]:
from dataclasses import replace

from config import EffortExperimentConfig

cfg = EffortExperimentConfig.default()
cfg = replace(
    cfg,
    graph=replace(cfg.graph, n_nodes=70000, graph_type="ba", ba_m=3),
)

torch.manual_seed(cfg.graph.seed)
np.random.seed(cfg.graph.seed)

# True parameters
TRUE_GAMMA = cfg.true_params.gamma
TRUE_LAMBDA = cfg.true_params.lambda_
TRUE_MU = cfg.true_params.mu
TRUE_R = cfg.true_params.r
TRUE_SIGMA_SQ = cfg.true_params.sigma_sq

# Graph
GRAPH_TYPE = cfg.graph.graph_type
N_NODES_TARGET = cfg.graph.n_nodes
BA_M = cfg.graph.ba_m
GRAPH_SEED = cfg.graph.seed

# Estimation
K = cfg.model.k
BATCH_SIZE = cfg.training.batch_size
N_DISC = cfg.training.n_disc
N_STEPS = cfg.training.n_steps
LR_D = cfg.training.lr_d
LR_G = cfg.training.lr_g
GRAD_CLIP_NORM_G = cfg.training.grad_clip_norm_g
HIDDEN_DIM = cfg.model.hidden_dim
DISC_NUM_LAYERS = cfg.model.resolved_discriminator_layers()
LOGIT_CLIP = cfg.model.logit_clip

# Picard/Newton iteration
LAMBDA_MAX = cfg.model.lambda_max
PICARD_TOL = cfg.model.picard_tol
PICARD_MAX = cfg.model.picard_max
NEWTON_TOL = cfg.model.newton_tol
NEWTON_MAX = cfg.model.newton_max
FIX_R = cfg.model.fix_r

# Generator initialization
INIT_GAMMA = cfg.init_params.gamma
INIT_LAMBDA = cfg.init_params.lambda_
INIT_MU = cfg.init_params.mu
INIT_R = cfg.init_params.r
INIT_LOG_SIGMA_SQ = cfg.init_params.log_sigma_sq

# Root sampling
ROOT_SAMPLER_MODE = cfg.training.resolved_root_sampler_mode()
ROOT_EXCLUSION_R = cfg.training.root_exclusion_r
DISJOINT_RESTARTS_K = cfg.training.resolved_disjoint_restarts_k()
DISJOINT_MIN_BATCH = cfg.training.resolved_disjoint_min_batch()
DISJOINT_RELAX_SEQUENCE = cfg.training.resolved_disjoint_relax_sequence()
DISJOINT_FALLBACK = cfg.training.disjoint_fallback
MIX_P_UNIFORM = cfg.training.mix_p_uniform
ROOT_SAMPLER_IS_DISJOINT = ROOT_SAMPLER_MODE != "uniform"

# Optional discriminator-input blur (instance noise)
INSTANCE_NOISE = cfg.instance_noise
if INSTANCE_NOISE.enabled and INSTANCE_NOISE.apply_to == "real_only":
    warnings.warn(
        "instance_noise.apply_to='real_only' is non-default and distorts only real discriminator targets.",
        RuntimeWarning,
    )

# Runtime diagnostics populated in later cells.
N_NODES = N_NODES_TARGET
graph_runtime_info = {}
sampling_calibration = {}
sampling_call_records = []
sampling_fallback_count = 0
sampling_call_counter = 0
last_sampling_info = None

RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = repo_root / "artifacts" / "effort_game_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

artifact_paths = []

history: dict[str, list[float]] = {
    "gamma": [],
    "lambda_": [],
    "mu": [],
    "r": [],
    "sigma_sq": [],
    "loss_d": [],
    "loss_g": [],
    "tau_x": [],
    "tau_y": [],
    "picard_iters": [],
    "newton_iters": [],
}

print("run_id:", RUN_ID)
print("run_dir:", RUN_DIR)
print("graph_type:", GRAPH_TYPE)
print("n_nodes override for validation:", N_NODES_TARGET)
print("ego_radius_k:", K, "disc_layers:", DISC_NUM_LAYERS, "root_exclusion_r:", ROOT_EXCLUSION_R)
print("root_sampler_mode:", ROOT_SAMPLER_MODE)
print("disjoint_restarts_k:", DISJOINT_RESTARTS_K)
print("disjoint_min_batch:", DISJOINT_MIN_BATCH)
print("disjoint_relax_sequence:", DISJOINT_RELAX_SEQUENCE)
print("generator_grad_clip_norm:", GRAD_CLIP_NORM_G)
print("instance_noise:", INSTANCE_NOISE)


run_id: 20260302_193428
run_dir: c:\Users\vitil\OneDrive\Desktop\adversarial_networks\artifacts\effort_game_runs\20260302_193428
graph_type: ba
n_nodes override for validation: 70000
ego_radius_k: 2 disc_layers: 2 root_exclusion_r: 4
root_sampler_mode: disjoint_best_of_k
disjoint_restarts_k: 50
disjoint_min_batch: 64
disjoint_relax_sequence: (3, 2)
generator_grad_clip_norm: 25.0
instance_noise: InstanceNoiseConfig(enabled=True, tau_x0=1.0, tau_y0=1.5, schedule='linear', anneal_steps=2500, min_tau=0.0, apply_to='both')


## C3 - Graph generation, W matrix, and ego-cache


In [3]:
start_time = time.time()


def _build_lfr_graph_with_retries(graph_cfg, max_attempts: int = 3) -> tuple[nx.Graph, int]:
    last_error = None
    for attempt in range(max_attempts):
        attempt_seed = graph_cfg.seed + attempt
        lfr_kwargs = {
            "n": graph_cfg.n_nodes,
            "tau1": graph_cfg.lfr_tau1,
            "tau2": graph_cfg.lfr_tau2,
            "mu": graph_cfg.lfr_mu,
            "min_community": graph_cfg.lfr_min_community,
            "max_community": graph_cfg.lfr_max_community,
            "max_degree": graph_cfg.lfr_max_degree,
            "seed": attempt_seed,
            "max_iters": graph_cfg.lfr_max_iters,
        }
        if graph_cfg.lfr_average_degree is not None:
            lfr_kwargs["average_degree"] = graph_cfg.lfr_average_degree
        else:
            lfr_kwargs["min_degree"] = graph_cfg.lfr_min_degree

        try:
            graph_raw = nx.generators.community.LFR_benchmark_graph(**lfr_kwargs)
            return nx.Graph(graph_raw), attempt + 1
        except nx.ExceededMaxIterations as exc:
            last_error = exc
            warnings.warn(
                "LFR generation exceeded max_iters "
                f"(attempt {attempt + 1}/{max_attempts}, seed={attempt_seed}, "
                f"max_iters={graph_cfg.lfr_max_iters}). Retrying.",
                RuntimeWarning,
            )

    raise RuntimeError(
        f"LFR generation failed after {max_attempts} attempts "
        f"(max_iters={graph_cfg.lfr_max_iters})."
    ) from last_error


if GRAPH_TYPE == "ba":
    G_nx = nx.barabasi_albert_graph(n=N_NODES_TARGET, m=BA_M, seed=GRAPH_SEED)
    lfr_attempts = None
else:
    G_nx, lfr_attempts = _build_lfr_graph_with_retries(cfg.graph, max_attempts=3)

n_nodes_before_sanitize = G_nx.number_of_nodes()
n_selfloops_removed = nx.number_of_selfloops(G_nx)
if n_selfloops_removed > 0:
    G_nx.remove_edges_from(nx.selfloop_edges(G_nx))

if not nx.is_connected(G_nx):
    gcc_nodes = max(nx.connected_components(G_nx), key=len)
    gcc_fraction = len(gcc_nodes) / n_nodes_before_sanitize
    G_nx = G_nx.subgraph(gcc_nodes).copy()
else:
    gcc_fraction = 1.0

if set(G_nx.nodes()) != set(range(G_nx.number_of_nodes())):
    G_nx = nx.convert_node_labels_to_integers(G_nx, first_label=0, ordering="sorted")

isolates = [node for node, deg_val in G_nx.degree() if deg_val == 0]
if isolates:
    raise RuntimeError(
        f"Graph has {len(isolates)} isolates after sanitization; cannot build row-stochastic W."
    )

N_NODES = G_nx.number_of_nodes()
edge_pairs = np.asarray(list(G_nx.edges()), dtype=np.int64)
if edge_pairs.size == 0:
    raise RuntimeError("Graph has no edges after sanitization.")

edge_index = torch.as_tensor(edge_pairs.T, dtype=torch.long)
edge_index = to_undirected(edge_index, num_nodes=N_NODES).contiguous().to(DEVICE)
W = build_row_stochastic_W(edge_index=edge_index, num_nodes=N_NODES).to(DEVICE)

deg = degree(edge_index[0], num_nodes=N_NODES, dtype=torch.float32)

graph_runtime_info = {
    "graph_type": GRAPH_TYPE,
    "seed": int(GRAPH_SEED),
    "configured_n_nodes": int(N_NODES_TARGET),
    "n_nodes": int(N_NODES),
    "n_edges": int(G_nx.number_of_edges()),
    "degree_min": float(deg.min().item()),
    "degree_mean": float(deg.mean().item()),
    "degree_max": float(deg.max().item()),
    "n_selfloops_removed": int(n_selfloops_removed),
    "gcc_fraction": float(gcc_fraction),
}
if GRAPH_TYPE == "ba":
    graph_runtime_info["ba_m"] = int(BA_M)
else:
    graph_runtime_info["lfr_params"] = {
        "tau1": float(cfg.graph.lfr_tau1),
        "tau2": float(cfg.graph.lfr_tau2),
        "mu": float(cfg.graph.lfr_mu),
        "average_degree": (
            int(cfg.graph.lfr_average_degree)
            if cfg.graph.lfr_average_degree is not None
            else None
        ),
        "min_degree": (
            int(cfg.graph.lfr_min_degree)
            if cfg.graph.lfr_min_degree is not None
            else None
        ),
        "min_community": int(cfg.graph.lfr_min_community),
        "max_community": int(cfg.graph.lfr_max_community),
        "max_degree": (
            int(cfg.graph.lfr_max_degree) if cfg.graph.lfr_max_degree is not None else None
        ),
        "max_iters": int(cfg.graph.lfr_max_iters),
        "attempts": int(lfr_attempts),
    }

cache_start = time.time()
ego_cache = {}
for root in range(N_NODES):
    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        node_idx=root,
        num_hops=K,
        edge_index=edge_index,
        relabel_nodes=True,
        num_nodes=N_NODES,
    )
    ego_cache[root] = (
        subset.to(DEVICE),
        sub_edge_index.to(DEVICE),
        int(mapping.item()),
    )
cache_seconds = time.time() - cache_start

root_sampler_rng = np.random.default_rng(GRAPH_SEED + 1)
uniform_sampler = RootSampler(
    num_nodes=N_NODES,
    mode="uniform",
    rng=root_sampler_rng,
)

if ROOT_SAMPLER_IS_DISJOINT:
    adjacency = build_adjacency_from_edge_index(edge_index=edge_index.cpu(), num_nodes=N_NODES)
    root_sampler = RootSampler(
        num_nodes=N_NODES,
        mode=ROOT_SAMPLER_MODE,
        exclusion_r=ROOT_EXCLUSION_R,
        disjoint_restarts_k=DISJOINT_RESTARTS_K,
        disjoint_min_batch=DISJOINT_MIN_BATCH,
        disjoint_relax_sequence=DISJOINT_RELAX_SEQUENCE,
        disjoint_fallback=DISJOINT_FALLBACK,
        rng=root_sampler_rng,
        adjacency=adjacency,
    )

    calibration_rng = np.random.default_rng(GRAPH_SEED + 2)
    calibration_sampler = RootSampler(
        num_nodes=N_NODES,
        mode=ROOT_SAMPLER_MODE,
        exclusion_r=ROOT_EXCLUSION_R,
        disjoint_restarts_k=DISJOINT_RESTARTS_K,
        disjoint_min_batch=DISJOINT_MIN_BATCH,
        disjoint_relax_sequence=DISJOINT_RELAX_SEQUENCE,
        disjoint_fallback=DISJOINT_FALLBACK,
        rng=calibration_rng,
        adjacency=adjacency,
    )
    calibration_sizes = np.array(
        [
            calibration_sampler.sample(batch_size=N_NODES).achieved_size
            for _ in range(50)
        ],
        dtype=np.int32,
    )
    sampling_calibration = {
        "trials": int(calibration_sizes.size),
        "min": int(calibration_sizes.min()),
        "p10": float(np.quantile(calibration_sizes, 0.10)),
        "median": float(np.quantile(calibration_sizes, 0.50)),
        "p90": float(np.quantile(calibration_sizes, 0.90)),
        "max": int(calibration_sizes.max()),
    }

    print(
        f"{ROOT_SAMPLER_MODE} calibration: "
        f"min={sampling_calibration['min']} "
        f"p10={sampling_calibration['p10']:.1f} "
        f"median={sampling_calibration['median']:.1f} "
        f"p90={sampling_calibration['p90']:.1f} "
        f"max={sampling_calibration['max']}"
    )

    if sampling_calibration["p10"] < DISJOINT_MIN_BATCH:
        warnings.warn(
            "Disjoint calibration p10 is below disjoint_min_batch: "
            f"p10={sampling_calibration['p10']:.1f}, "
            f"disjoint_min_batch={DISJOINT_MIN_BATCH}.",
            RuntimeWarning,
        )
else:
    root_sampler = uniform_sampler

print(f"nodes: {graph_runtime_info['n_nodes']}")
print(f"edges (undirected): {graph_runtime_info['n_edges']}")
print(
    "degree mean/min/max: "
    f"{graph_runtime_info['degree_mean']:.3f} / "
    f"{graph_runtime_info['degree_min']:.0f} / "
    f"{graph_runtime_info['degree_max']:.0f}"
)
print(f"self-loops removed: {graph_runtime_info['n_selfloops_removed']}")
print(f"gcc_fraction: {graph_runtime_info['gcc_fraction']:.3f}")
print(f"ego-cache entries: {len(ego_cache)}")
print(f"ego-cache build time: {cache_seconds:.2f}s")
print(f"C3 total time: {time.time() - start_time:.2f}s")


disjoint_best_of_k calibration: min=361 p10=365.0 median=369.5 p90=376.1 max=380
nodes: 70000
edges (undirected): 209991
degree mean/min/max: 6.000 / 3 / 981
self-loops removed: 0
gcc_fraction: 1.000
ego-cache entries: 70000
ego-cache build time: 188.00s
C3 total time: 1351.34s


## C4 - Simulate observed equilibrium


In [4]:
X = torch.randn(N_NODES, device=DEVICE)

true_generator = EffortGameGenerator(
    lambda_max=LAMBDA_MAX,
    picard_tol=PICARD_TOL,
    picard_max=PICARD_MAX,
    newton_tol=NEWTON_TOL,
    newton_max=NEWTON_MAX,
    fix_r=TRUE_R,
    init_gamma=TRUE_GAMMA,
    init_lambda=TRUE_LAMBDA,
    init_mu=TRUE_MU,
    init_r=TRUE_R,
    init_log_sigma_sq=math.log(TRUE_SIGMA_SQ),
).to(DEVICE)

with torch.no_grad():
    Y_obs = true_generator(W, X)
obs_picard_iters = true_generator.last_picard_iterations
obs_newton_iters = true_generator.last_newton_max_iters

norm_stats = {
    "mu_X": float(X.mean().item()),
    "sigma_X": float(X.std(unbiased=False).item()),
    "mu_Y": float(Y_obs.mean().item()),
    "sigma_Y": float(Y_obs.std(unbiased=False).item()),
}
assert norm_stats["sigma_X"] > 0.0
assert norm_stats["sigma_Y"] > 0.0

print("observed Picard iterations:", obs_picard_iters)
print("observed Newton max iterations:", obs_newton_iters)
print("normalization constants:", norm_stats)


observed Picard iterations: 100
observed Newton max iterations: 8
normalization constants: {'mu_X': -0.00043101568007841706, 'sigma_X': 1.0063687562942505, 'mu_Y': 0.45643511414527893, 'sigma_Y': 0.909787118434906}


## C5 - Exploratory data analysis (Figure 1, Table 1)


In [5]:
def write_csv_rows(path: Path, header: list[str], rows: list[list[object]]) -> None:
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        writer.writerows(rows)


def _sample_skew_kurtosis(values: np.ndarray) -> tuple[float, float]:
    try:
        from scipy.stats import kurtosis as scipy_kurtosis
        from scipy.stats import skew as scipy_skew

        skew_val = float(scipy_skew(values, bias=False))
        kurt_val = float(scipy_kurtosis(values, fisher=False, bias=False))
        return skew_val, kurt_val
    except Exception:
        centered = values - float(values.mean())
        std = float(values.std())
        if std <= 0.0:
            return 0.0, 3.0
        z = centered / std
        return float(np.mean(z**3)), float(np.mean(z**4))


deg_np = deg.cpu().numpy()
X_np = X.detach().cpu().numpy()
Y_obs_np = Y_obs.detach().cpu().numpy()
ols_slope, ols_intercept = np.polyfit(X_np, Y_obs_np, deg=1)
skew_y, kurt_y = _sample_skew_kurtosis(Y_obs_np)

fig1, axes = plt.subplots(
    1,
    3,
    figsize=figure_size("double", n_rows=2, n_cols=3),
    constrained_layout=True,
)

_, _, patches_deg = axes[0].hist(
    deg_np,
    bins=35,
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_deg), hatch_index=0, facecolor="white")
axes[0].set_yscale("log")
axes[0].set_title("(a) Degree Distribution")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count (log scale)")
style_axes(axes[0])

_, _, patches_y = axes[1].hist(
    Y_obs_np,
    bins=40,
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_y), hatch_index=1, facecolor="0.9")
axes[1].set_title("(b) Observed Outcome Distribution")
axes[1].set_xlabel("Y_obs")
axes[1].set_ylabel("Count")
style_axes(axes[1])

axes[2].scatter(X_np, Y_obs_np, s=10, **scatter_style(0))
x_line = np.linspace(X_np.min(), X_np.max(), 200)
axes[2].plot(
    x_line,
    ols_slope * x_line + ols_intercept,
    label=f"OLS slope={ols_slope:.3f}",
    **line_style(1, with_markers=False),
)
axes[2].set_title("(c) X vs Y_obs")
axes[2].set_xlabel("X")
axes[2].set_ylabel("Y_obs")
style_axes(axes[2])
axes[2].legend(loc="best")

fig1_path = RUN_DIR / "fig01_observed_data.png"
fig1.savefig(fig1_path, dpi=FIGURE_DPI, format="png")
plt.close(fig1)

rows_table1 = [
    ["Nodes", int(N_NODES)],
    ["Edges (undirected)", int(G_nx.number_of_edges())],
    ["Mean degree", float(deg.mean().item())],
    ["Max degree", float(deg.max().item())],
    ["Mean Y_obs", float(Y_obs.mean().item())],
    ["Std Y_obs", float(Y_obs.std(unbiased=False).item())],
    ["Skewness Y_obs", skew_y],
    ["Kurtosis Y_obs", kurt_y],
    ["OLS slope (Y on X)", float(ols_slope)],
    ["Picard iterations", int(obs_picard_iters)],
    ["Newton max iterations", int(obs_newton_iters)],
    ["True lambda", float(TRUE_LAMBDA)],
    ["True mu", float(TRUE_MU)],
    ["True r", float(TRUE_R)],
    ["Contraction rate rho", float(TRUE_LAMBDA / (1.0 + TRUE_LAMBDA))],
]
tab1_path = RUN_DIR / "tab01_data_summary.csv"
write_csv_rows(tab1_path, ["Statistic", "Value"], rows_table1)

artifact_paths.extend([fig1_path, tab1_path])

plot_ground_truth_graph_outcomes(
    graph=G_nx,
    outcomes=Y_obs_np,
    save_path=RUN_DIR / "fig07_ground_truth_graph_outcomes.png",
    max_nodes=1200,
    layout_seed=GRAPH_SEED,
)
fig7_path = RUN_DIR / "fig07_ground_truth_graph_outcomes.png"
artifact_paths.append(fig7_path)
print("saved:", fig7_path.name)

print("Table 1")
for key, value in rows_table1:
    print(f"  {key}: {value}")


saved: fig07_ground_truth_graph_outcomes.png
Table 1
  Nodes: 70000
  Edges (undirected): 209991
  Mean degree: 5.9997429847717285
  Max degree: 981.0
  Mean Y_obs: 0.45643511414527893
  Std Y_obs: 0.909787118434906
  Skewness Y_obs: 0.4275458245160849
  Kurtosis Y_obs: 3.034531024394707
  OLS slope (Y on X): 0.7381206692224225
  Picard iterations: 100
  Newton max iterations: 8
  True lambda: 0.6666666666666666
  True mu: 0.5
  True r: 1.0
  Contraction rate rho: 0.4


## C6 - Instantiate models and optimizers


In [6]:
generator = EffortGameGenerator(
    lambda_max=LAMBDA_MAX,
    picard_tol=PICARD_TOL,
    picard_max=PICARD_MAX,
    newton_tol=NEWTON_TOL,
    newton_max=NEWTON_MAX,
    fix_r=FIX_R,
    init_gamma=INIT_GAMMA,
    init_lambda=INIT_LAMBDA,
    init_mu=INIT_MU,
    init_r=INIT_R,
    init_log_sigma_sq=INIT_LOG_SIGMA_SQ,
).to(DEVICE)

discriminator = RootedMPNNDiscriminator(
    hidden_dim=HIDDEN_DIM,
    num_layers=DISC_NUM_LAYERS,
    logit_clip=LOGIT_CLIP,
).to(DEVICE)

opt_d = torch.optim.Adam(discriminator.parameters(), lr=LR_D)
opt_g = torch.optim.Adam(generator.parameters(), lr=LR_G)

num_gen_params = sum(param.numel() for param in generator.parameters())
num_disc_params = sum(param.numel() for param in discriminator.parameters())

print("generator parameter count:", num_gen_params)
print("discriminator parameter count:", num_disc_params)
print("initial theta:", generator.get_params())


generator parameter count: 3
discriminator parameter count: 17219
initial theta: {'gamma': 0.0, 'lambda_': 0.5000000596046448, 'mu': 0.09999999403953552, 'r': 1.0, 'sigma_sq': 1.0}


## C7 - Training loop (detach discipline + logging)


In [7]:
def sample_roots(batch_size: int) -> torch.Tensor:
    global sampling_call_counter, sampling_fallback_count, last_sampling_info

    sampling_call_counter += 1

    if ROOT_SAMPLER_IS_DISJOINT and MIX_P_UNIFORM > 0.0 and root_sampler_rng.random() < MIX_P_UNIFORM:
        roots, result = sample_roots_tensor(
            sampler=uniform_sampler,
            batch_size=batch_size,
            device=DEVICE,
        )
        record = {
            "call_index": int(sampling_call_counter),
            "requested_size": int(batch_size),
            "achieved_size": int(batch_size),
            "mode": "uniform_mix",
            "r_used": 0,
            "attempts_used": int(result.attempts_used),
            "met_target": True,
            "fallback_reason": "mix_p_uniform",
        }
    else:
        roots, result = sample_roots_tensor(
            sampler=root_sampler,
            batch_size=batch_size,
            device=DEVICE,
        )
        record = {
            "call_index": int(sampling_call_counter),
            "requested_size": int(batch_size),
            "achieved_size": int(result.achieved_size),
            "mode": str(result.mode),
            "r_used": int(result.radius_used) if result.radius_used is not None else 0,
            "attempts_used": int(result.attempts_used),
            "met_target": bool(result.met_target),
            "fallback_reason": str(result.fallback_reason),
        }

    if ROOT_SAMPLER_IS_DISJOINT and record["fallback_reason"] not in {"", "mix_p_uniform"}:
        sampling_fallback_count += 1

    sampling_call_records.append(record)
    last_sampling_info = record
    return roots


train_start = time.time()
for step in range(1, N_STEPS + 1):
    if step in (1000, 1500):
        for group in opt_g.param_groups:
            group["lr"] *= 0.5

    discriminator.train()
    for param in discriminator.parameters():
        param.requires_grad_(True)

    with torch.no_grad():
        Y_sim_detached = generator(W, X)

    tau_x_step, tau_y_step = compute_instance_noise_taus(
        instance_noise=INSTANCE_NOISE,
        generator_step=step,
    )

    step_sampling_achieved = []
    step_sampling_attempts = []

    last_loss_d = float("nan")
    for _ in range(N_DISC):
        roots = sample_roots(BATCH_SIZE)
        step_sampling_achieved.append(int(last_sampling_info["achieved_size"]))
        step_sampling_attempts.append(int(last_sampling_info["attempts_used"]))

        batch_obs, root_idx_obs = extract_ego_batch(
            roots=roots,
            ego_cache=ego_cache,
            X=X,
            Y=Y_obs,
            norm_stats=norm_stats,
            instance_noise=INSTANCE_NOISE,
            generator_step=step,
            batch_role="real",
        )

        batch_sim, root_idx_sim = extract_ego_batch(
            roots=roots,
            ego_cache=ego_cache,
            X=X,
            Y=Y_sim_detached,
            norm_stats=norm_stats,
            instance_noise=INSTANCE_NOISE,
            generator_step=step,
            batch_role="fake",
        )

        opt_d.zero_grad(set_to_none=True)
        logits_real = discriminator(batch_obs.x, batch_obs.edge_index, root_idx_obs)
        logits_fake = discriminator(batch_sim.x, batch_sim.edge_index, root_idx_sim)
        loss_d = F.softplus(-logits_real).mean() + F.softplus(logits_fake).mean()
        loss_d.backward()
        opt_d.step()
        last_loss_d = float(loss_d.item())

    discriminator.train()
    for param in discriminator.parameters():
        param.requires_grad_(False)

    roots_g = sample_roots(BATCH_SIZE)
    step_sampling_achieved.append(int(last_sampling_info["achieved_size"]))
    step_sampling_attempts.append(int(last_sampling_info["attempts_used"]))

    Y_sim = generator(W, X)
    batch_sim_g, root_idx_sim_g = extract_ego_batch(
        roots=roots_g,
        ego_cache=ego_cache,
        X=X,
        Y=Y_sim,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=step,
        batch_role="fake",
    )

    opt_g.zero_grad(set_to_none=True)
    logits_fake_g = discriminator(batch_sim_g.x, batch_sim_g.edge_index, root_idx_sim_g)
    loss_g = F.softplus(-logits_fake_g).mean()
    loss_g.backward()
    nn.utils.clip_grad_norm_(generator.parameters(), max_norm=GRAD_CLIP_NORM_G)
    opt_g.step()

    theta = generator.get_params()
    history["gamma"].append(theta["gamma"])
    history["lambda_"].append(theta["lambda_"])
    history["mu"].append(theta["mu"])
    history["r"].append(theta["r"])
    history["sigma_sq"].append(theta["sigma_sq"])
    history["loss_d"].append(last_loss_d)
    history["loss_g"].append(float(loss_g.item()))
    history["tau_x"].append(float(tau_x_step))
    history["tau_y"].append(float(tau_y_step))
    history["picard_iters"].append(float(generator.last_picard_iterations))
    history["newton_iters"].append(float(generator.last_newton_max_iters))

    if step == 1 or step % 50 == 0:
        log_msg = (
            f"step {step:4d}/{N_STEPS} | "
            f"gamma={theta['gamma']:.4f} lambda={theta['lambda_']:.4f} "
            f"mu={theta['mu']:.4f} r={theta['r']:.4f} sigma_sq={theta['sigma_sq']:.4f} | "
            f"picard={generator.last_picard_iterations:3d} newton={generator.last_newton_max_iters:2d} | "
            f"tau_x={tau_x_step:.4f} tau_y={tau_y_step:.4f} | "
            f"L_D={last_loss_d:.4f} L_G={loss_g.item():.4f}"
        )
        if ROOT_SAMPLER_IS_DISJOINT and step_sampling_achieved:
            achieved_min = int(np.min(step_sampling_achieved))
            achieved_mean = float(np.mean(step_sampling_achieved))
            tries_mean = float(np.mean(step_sampling_attempts))
            tries_max = int(np.max(step_sampling_attempts))
            log_msg += (
                f" | |S| min/mean={achieved_min}/{achieved_mean:.1f} of {BATCH_SIZE} "
                f"tries(mean/max)={tries_mean:.2f}/{tries_max}"
            )
        print(log_msg)

train_seconds = time.time() - train_start
print(f"training time: {train_seconds:.2f}s")


step    1/2000 | gamma=-0.0070 lambda=0.5031 mu=0.1007 r=1.0000 sigma_sq=1.0000 | picard= 20 newton= 8 | tau_x=0.9996 tau_y=1.4994 | L_D=1.4113 L_G=0.8044 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step   50/2000 | gamma=-0.0221 lambda=0.3778 mu=0.1494 r=1.0000 sigma_sq=1.0000 | picard= 18 newton= 8 | tau_x=0.9800 tau_y=1.4700 | L_D=1.0384 L_G=0.8802 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  100/2000 | gamma=0.4679 lambda=0.2367 mu=0.2654 r=1.0000 sigma_sq=1.0000 | picard= 15 newton= 8 | tau_x=0.9600 tau_y=1.4400 | L_D=1.1256 L_G=1.9740 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  150/2000 | gamma=1.1354 lambda=0.1332 mu=0.5095 r=1.0000 sigma_sq=1.0000 | picard=100 newton= 8 | tau_x=0.9400 tau_y=1.4100 | L_D=2.0281 L_G=0.6056 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  200/2000 | gamma=1.3525 lambda=0.1142 mu=0.6239 r=1.0000 sigma_sq=1.0000 | picard=100 newton= 8 | tau_x=0.9200 tau_y=1.3800 | L_D=1.3485 L_G=0.7591 | |S| min/mean=64/64

## C8 - Parameter convergence (Figure 2)


In [8]:
steps = np.arange(1, N_STEPS + 1)

series = [
    ("gamma", TRUE_GAMMA, r"$\gamma$"),
    ("lambda_", TRUE_LAMBDA, r"$\lambda$"),
    ("mu", TRUE_MU, r"$\mu$"),
    ("sigma_sq", TRUE_SIGMA_SQ, r"$\sigma^2$"),
]
if FIX_R is None:
    series.insert(3, ("r", TRUE_R, "r"))

fig2, axes = plt.subplots(
    len(series),
    1,
    figsize=figure_size("single", n_rows=len(series), n_cols=1),
    sharex=True,
    constrained_layout=True,
)
if len(series) == 1:
    axes = [axes]

markevery = max(1, N_STEPS // 18)
for ax, (key, true_value, label) in zip(axes, series):
    ax.plot(steps, history[key], markevery=markevery, **series_style(0))
    add_reference_line(ax, value=true_value, text=f"true={true_value:.3g}")
    ax.set_ylabel(label)
    style_axes(ax)

axes[-1].set_xlabel("Generator Step")

fig2_path = RUN_DIR / "fig02_theta_convergence.png"
fig2.savefig(fig2_path, dpi=FIGURE_DPI, format="png")
plt.close(fig2)
artifact_paths.append(fig2_path)

print("saved:", fig2_path.name)


saved: fig02_theta_convergence.png


## C9 - Loss convergence and discriminator scores (Figures 3-4)


In [9]:
def rolling_mean(values: list[float], window: int) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=np.float64)
    if arr.size < window:
        return np.arange(1, arr.size + 1), arr
    kernel = np.ones(window, dtype=np.float64) / float(window)
    smooth = np.convolve(arr, kernel, mode="valid")
    x_axis = np.arange(window, arr.size + 1)
    return x_axis, smooth


plot_loss_convergence(
    loss_d_history=history["loss_d"],
    loss_g_history=history["loss_g"],
    save_path=RUN_DIR / "fig03_loss_convergence.png",
    window=50,
)
fig3_path = RUN_DIR / "fig03_loss_convergence.png"
artifact_paths.append(fig3_path)

print("saved:", fig3_path.name)


discriminator.eval()
with torch.no_grad():
    roots_diag = sample_roots(512)
    batch_real, root_real = extract_ego_batch(
        roots=roots_diag,
        ego_cache=ego_cache,
        X=X,
        Y=Y_obs,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=N_STEPS,
        batch_role="real",
    )

    Y_sim_diag = generator(W, X)
    batch_fake, root_fake = extract_ego_batch(
        roots=roots_diag,
        ego_cache=ego_cache,
        X=X,
        Y=Y_sim_diag,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=N_STEPS,
        batch_role="fake",
    )

    logits_real = discriminator(batch_real.x, batch_real.edge_index, root_real)
    logits_fake = discriminator(batch_fake.x, batch_fake.edge_index, root_fake)
    scores_real = torch.sigmoid(logits_real).cpu().numpy()
    scores_fake = torch.sigmoid(logits_fake).cpu().numpy()

fig4, ax = plt.subplots(
    figsize=figure_size("single", n_rows=1, n_cols=1),
    constrained_layout=True,
)
_, _, patches_real = ax.hist(
    scores_real,
    bins=30,
    density=True,
    label="Real",
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_real), hatch_index=0, facecolor="0.9")
_, _, patches_fake = ax.hist(
    scores_fake,
    bins=30,
    density=True,
    label="Fake",
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_fake), hatch_index=1, facecolor="white")
add_reference_line(ax, value=0.5, text="0.5", axis="x")
ax.set_title("Figure 4: Discriminator Scores")
ax.set_xlabel("D(score)")
ax.set_ylabel("Density")
style_axes(ax)
ax.legend(loc="best")

fig4_path = RUN_DIR / "fig04_discriminator_scores.png"
fig4.savefig(fig4_path, dpi=FIGURE_DPI, format="png")
plt.close(fig4)
artifact_paths.append(fig4_path)

print("saved:", fig4_path.name)


saved: fig03_loss_convergence.png
saved: fig04_discriminator_scores.png


## C10 - Final estimation results (Table 2)


In [10]:
theta_hat = generator.get_params()

rows_table2 = [
    ["gamma", TRUE_GAMMA, theta_hat["gamma"], abs(theta_hat["gamma"] - TRUE_GAMMA)],
    ["lambda_", TRUE_LAMBDA, theta_hat["lambda_"], abs(theta_hat["lambda_"] - TRUE_LAMBDA)],
    ["mu", TRUE_MU, theta_hat["mu"], abs(theta_hat["mu"] - TRUE_MU)],
    ["r", TRUE_R, theta_hat["r"], abs(theta_hat["r"] - TRUE_R)],
    ["sigma_sq", TRUE_SIGMA_SQ, theta_hat["sigma_sq"], abs(theta_hat["sigma_sq"] - TRUE_SIGMA_SQ)],
]

tab2_path = RUN_DIR / "tab02_estimation_results.csv"
write_csv_rows(
    tab2_path,
    ["Parameter", "True", "Estimated", "Absolute Error"],
    rows_table2,
)
artifact_paths.append(tab2_path)

print("Table 2")
for row in rows_table2:
    print(f"  {row[0]:>8s} | true={row[1]:.4f} est={row[2]:.4f} abs_err={row[3]:.4f}")


Table 2
     gamma | true=1.5000 est=1.6398 abs_err=0.1398
   lambda_ | true=0.6667 est=0.8134 abs_err=0.1467
        mu | true=0.5000 est=0.5095 abs_err=0.0095
         r | true=1.0000 est=1.0000 abs_err=0.0000
  sigma_sq | true=1.0000 est=1.0000 abs_err=0.0000


## C11 - Picard/Newton iteration diagnostics


In [11]:
fig8, axes = plt.subplots(
    2,
    1,
    figsize=figure_size("single", n_rows=2, n_cols=1),
    sharex=True,
    constrained_layout=True,
)

steps = np.arange(1, N_STEPS + 1)
axes[0].plot(steps, history["picard_iters"], **series_style(0))
axes[0].set_ylabel("Picard iters")
axes[0].set_title("(a) Picard Iterations per Step")
style_axes(axes[0])

axes[1].plot(steps, history["newton_iters"], **series_style(1))
axes[1].set_ylabel("Newton iters")
axes[1].set_xlabel("Generator Step")
axes[1].set_title("(b) Newton Max Iterations per Step")
style_axes(axes[1])

fig8_path = RUN_DIR / "fig08_solver_iterations.png"
fig8.savefig(fig8_path, dpi=FIGURE_DPI, format="png")
plt.close(fig8)
artifact_paths.append(fig8_path)

print("saved:", fig8_path.name)


saved: fig08_solver_iterations.png


## C12 - Simulated vs observed outcome distributions (Figure 5)


In [12]:
with torch.no_grad():
    Y_sim_final = generator(W, X)

Y_sim_np = Y_sim_final.detach().cpu().numpy()

fig5, axes = plt.subplots(
    1,
    2,
    figsize=figure_size("double", n_rows=1, n_cols=2),
    constrained_layout=True,
)

_, _, patches_obs = axes[0].hist(
    Y_obs_np,
    bins=40,
    density=True,
    label="Y_obs",
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_obs), hatch_index=0, facecolor="0.9")
_, _, patches_sim = axes[0].hist(
    Y_sim_np,
    bins=40,
    density=True,
    label="Y_sim(theta_hat)",
    edgecolor="black",
    linewidth=0.6,
)
style_histogram_patches(list(patches_sim), hatch_index=1, facecolor="white")
axes[0].set_title("(a) Outcome Histograms")
axes[0].set_xlabel("Y")
axes[0].set_ylabel("Density")
style_axes(axes[0])
axes[0].legend(loc="best")

q_grid = np.linspace(0.01, 0.99, 99)
q_obs = np.quantile(Y_obs_np, q_grid)
q_sim = np.quantile(Y_sim_np, q_grid)
min_q = float(min(q_obs.min(), q_sim.min()))
max_q = float(max(q_obs.max(), q_sim.max()))
axes[1].scatter(q_obs, q_sim, s=14, **scatter_style(0))
axes[1].plot(
    [min_q, max_q],
    [min_q, max_q],
    **line_style(1, with_markers=False),
)
axes[1].set_title("(b) Q-Q Plot")
axes[1].set_xlabel("Observed Quantiles")
axes[1].set_ylabel("Simulated Quantiles")
style_axes(axes[1])

fig5_path = RUN_DIR / "fig05_Y_distributions.png"
fig5.savefig(fig5_path, dpi=FIGURE_DPI, format="png")
plt.close(fig5)
artifact_paths.append(fig5_path)

print("saved:", fig5_path.name)


saved: fig05_Y_distributions.png


## C13 - Convergence tail statistics (Table 3, Figure 6)


In [13]:
tail_window = min(500, N_STEPS)

gamma_tail = np.asarray(history["gamma"][-tail_window:], dtype=np.float64)
lambda_tail = np.asarray(history["lambda_"][-tail_window:], dtype=np.float64)
mu_tail = np.asarray(history["mu"][-tail_window:], dtype=np.float64)
r_tail = np.asarray(history["r"][-tail_window:], dtype=np.float64)
sigma_tail = np.asarray(history["sigma_sq"][-tail_window:], dtype=np.float64)

tail_rows = [
    ["gamma", float(gamma_tail.mean()), float(gamma_tail.std(ddof=0)), TRUE_GAMMA],
    ["lambda_", float(lambda_tail.mean()), float(lambda_tail.std(ddof=0)), TRUE_LAMBDA],
    ["mu", float(mu_tail.mean()), float(mu_tail.std(ddof=0)), TRUE_MU],
    ["r", float(r_tail.mean()), float(r_tail.std(ddof=0)), TRUE_R],
    ["sigma_sq", float(sigma_tail.mean()), float(sigma_tail.std(ddof=0)), TRUE_SIGMA_SQ],
]

tail_std = {
    "gamma": float(gamma_tail.std(ddof=0)),
    "lambda_": float(lambda_tail.std(ddof=0)),
    "mu": float(mu_tail.std(ddof=0)),
    "r": float(r_tail.std(ddof=0)),
    "sigma_sq": float(sigma_tail.std(ddof=0)),
}

tab3_path = RUN_DIR / "tab03_convergence_tail.csv"
write_csv_rows(
    tab3_path,
    ["Parameter", "Mean (last 500)", "Std (last 500)", "True"],
    tail_rows,
)
artifact_paths.append(tab3_path)

series = [
    ("gamma", TRUE_GAMMA, r"$\gamma$"),
    ("lambda_", TRUE_LAMBDA, r"$\lambda$"),
    ("mu", TRUE_MU, r"$\mu$"),
    ("r", TRUE_R, "r"),
    ("sigma_sq", TRUE_SIGMA_SQ, r"$\sigma^2$"),
]
tail_steps = np.arange(N_STEPS - tail_window + 1, N_STEPS + 1)
fig6, axes = plt.subplots(
    len(series),
    1,
    figsize=figure_size("single", n_rows=len(series), n_cols=1),
    sharex=True,
    constrained_layout=True,
)
if len(series) == 1:
    axes = [axes]

for ax, (key, true_value, label) in zip(axes, series):
    ax.plot(tail_steps, history[key][-tail_window:], **series_style(0))
    add_reference_line(ax, value=true_value, text=f"true={true_value:.3g}")
    ax.set_ylabel(label)
    style_axes(ax)

axes[-1].set_xlabel("Generator Step")

fig6_path = RUN_DIR / "fig06_tail_stability.png"
fig6.savefig(fig6_path, dpi=FIGURE_DPI, format="png")
plt.close(fig6)
artifact_paths.append(fig6_path)

print("Table 3")
for row in tail_rows:
    print(f"  {row[0]:>8s} | mean={row[1]:.4f} std={row[2]:.4f} true={row[3]:.4f}")


Table 3
     gamma | mean=1.5413 std=0.0456 true=1.5000
   lambda_ | mean=0.7921 std=0.0612 true=0.6667
        mu | mean=0.5255 std=0.0142 true=0.5000
         r | mean=1.0000 std=0.0000 true=1.0000
  sigma_sq | mean=1.0000 std=0.0000 true=1.0000


## C14 - Save all artifacts and write run manifest


In [14]:
def sha256_file(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


if ROOT_SAMPLER_IS_DISJOINT and sampling_call_records:
    sampling_stats_path = RUN_DIR / "sampling_roots_per_call.csv"
    with sampling_stats_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "call_index",
                "requested_size",
                "achieved_size",
                "mode",
                "r_used",
                "attempts_used",
                "met_target",
                "fallback_reason",
            ],
        )
        writer.writeheader()
        writer.writerows(sampling_call_records)
    artifact_paths.append(sampling_stats_path)

seen = set()
unique_artifacts = []
for path in artifact_paths:
    if path.name not in seen:
        unique_artifacts.append(path)
        seen.add(path.name)

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True, stderr=subprocess.DEVNULL
    ).strip()
    git_status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo_root, text=True, stderr=subprocess.DEVNULL
    )
    git_is_dirty = bool(git_status.strip())
except Exception:
    git_commit = None
    git_is_dirty = None

conda_lock_path = repo_root / "conda-lock.yml"
env_yml_path = repo_root / "environment.yml"

if ROOT_SAMPLER_IS_DISJOINT and sampling_call_records:
    achieved_arr = np.asarray([row["achieved_size"] for row in sampling_call_records], dtype=np.int32)
    attempts_arr = np.asarray([row["attempts_used"] for row in sampling_call_records], dtype=np.int32)
    sampling_runtime_stats = {
        "calls": int(achieved_arr.size),
        "min": int(achieved_arr.min()),
        "p10": float(np.quantile(achieved_arr, 0.10)),
        "median": float(np.quantile(achieved_arr, 0.50)),
        "p90": float(np.quantile(achieved_arr, 0.90)),
        "max": int(achieved_arr.max()),
        "attempts_mean": float(attempts_arr.mean()),
        "attempts_max": int(attempts_arr.max()),
        "fallback_count": int(sampling_fallback_count),
    }
else:
    sampling_runtime_stats = None

manifest = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "git": {"commit": git_commit, "is_dirty": git_is_dirty},
    "platform": {
        "os": platform.platform(),
        "python": platform.python_version(),
    },
    "versions": {
        "torch": torch.__version__,
        "torch_geometric": torch_geometric.__version__,
        "numpy": np.__version__,
        "networkx": nx.__version__,
    },
    "files": {
        "conda_lock_sha256": sha256_file(conda_lock_path) if conda_lock_path.exists() else None,
        "environment_yml_sha256": sha256_file(env_yml_path) if env_yml_path.exists() else None,
    },
    "inputs": {
        "graph": {
            "graph_type": cfg.graph.graph_type,
            "n_nodes": int(cfg.graph.n_nodes),
            "seed": int(cfg.graph.seed),
            "ba_m": int(cfg.graph.ba_m),
            "lfr_tau1": float(cfg.graph.lfr_tau1),
            "lfr_tau2": float(cfg.graph.lfr_tau2),
            "lfr_mu": float(cfg.graph.lfr_mu),
            "lfr_average_degree": (
                int(cfg.graph.lfr_average_degree)
                if cfg.graph.lfr_average_degree is not None
                else None
            ),
            "lfr_min_degree": (
                int(cfg.graph.lfr_min_degree) if cfg.graph.lfr_min_degree is not None else None
            ),
            "lfr_min_community": int(cfg.graph.lfr_min_community),
            "lfr_max_community": int(cfg.graph.lfr_max_community),
            "lfr_max_degree": (
                int(cfg.graph.lfr_max_degree) if cfg.graph.lfr_max_degree is not None else None
            ),
            "lfr_max_iters": int(cfg.graph.lfr_max_iters),
            "realized": graph_runtime_info,
        },
        "sampling": {
            "root_sampler_mode": ROOT_SAMPLER_MODE,
            "root_exclusion_r": int(ROOT_EXCLUSION_R),
            "disjoint_restarts_k": int(DISJOINT_RESTARTS_K),
            "disjoint_min_batch": int(DISJOINT_MIN_BATCH),
            "disjoint_relax_sequence": [int(v) for v in DISJOINT_RELAX_SEQUENCE],
            "disjoint_fallback": str(DISJOINT_FALLBACK),
            "mix_p_uniform": float(MIX_P_UNIFORM),
            "calibration": sampling_calibration if ROOT_SAMPLER_IS_DISJOINT else None,
            "runtime": sampling_runtime_stats,
        },
        "instance_noise": {
            "enabled": bool(INSTANCE_NOISE.enabled),
            "tau_x0": float(INSTANCE_NOISE.tau_x0),
            "tau_y0": float(INSTANCE_NOISE.tau_y0),
            "schedule": str(INSTANCE_NOISE.schedule),
            "anneal_steps": int(INSTANCE_NOISE.anneal_steps),
            "min_tau": float(INSTANCE_NOISE.min_tau),
            "apply_to": str(INSTANCE_NOISE.apply_to),
            "final_tau_x": float(history["tau_x"][-1]) if history["tau_x"] else 0.0,
            "final_tau_y": float(history["tau_y"][-1]) if history["tau_y"] else 0.0,
        },
        "model": {
            "k": K,
            "lambda_max": LAMBDA_MAX,
            "picard_tol": PICARD_TOL,
            "picard_max": PICARD_MAX,
            "newton_tol": NEWTON_TOL,
            "newton_max": NEWTON_MAX,
            "fix_r": FIX_R,
            "hidden_dim": HIDDEN_DIM,
            "logit_clip": LOGIT_CLIP,
        },
        "training": {
            "steps": N_STEPS,
            "n_disc": N_DISC,
            "batch_size": BATCH_SIZE,
            "lr_d": LR_D,
            "lr_g": LR_G,
        },
    },
    "truth": {
        "gamma0": TRUE_GAMMA,
        "lambda0": TRUE_LAMBDA,
        "mu0": TRUE_MU,
        "r0": TRUE_R,
        "sigma_sq0": TRUE_SIGMA_SQ,
    },
    "estimate": {
        "gamma_hat": float(theta_hat["gamma"]),
        "lambda_hat": float(theta_hat["lambda_"]),
        "mu_hat": float(theta_hat["mu"]),
        "r_hat": float(theta_hat["r"]),
        "sigma_sq_hat": float(theta_hat["sigma_sq"]),
    },
    "diagnostics": {
        "tail_std": tail_std,
        "final_picard_iters": float(history["picard_iters"][-1]) if history["picard_iters"] else 0.0,
        "final_newton_iters": float(history["newton_iters"][-1]) if history["newton_iters"] else 0.0,
    },
    "artifacts": [
        {"path": path.name, "sha256": sha256_file(path)}
        for path in unique_artifacts
    ],
}

manifest_path = RUN_DIR / "run_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print("Artifacts written:")
for path in sorted(RUN_DIR.glob("*")):
    print(f"  {path.name}: {path.stat().st_size} bytes")


Artifacts written:
  fig01_observed_data.png: 117871 bytes
  fig02_theta_convergence.png: 82582 bytes
  fig03_loss_convergence.png: 50765 bytes
  fig04_discriminator_scores.png: 30447 bytes
  fig05_Y_distributions.png: 53086 bytes
  fig06_tail_stability.png: 104294 bytes
  fig07_ground_truth_graph_outcomes.png: 184860 bytes
  fig08_solver_iterations.png: 48466 bytes
  run_manifest.json: 5119 bytes
  sampling_roots_per_call.csv: 163032 bytes
  tab01_data_summary.csv: 407 bytes
  tab02_estimation_results.csv: 243 bytes
  tab03_convergence_tail.csv: 249 bytes
